# M1.0e Primer: Charge, Coulomb's Law, Field and Potential

Book pointer: chapter 2, sections 2.1 (Coulomb's law), 2.2 (electric field), 2.3 (visualizing fields), 2.4 (continuous distributions); chapter 3, sections 3.1 (electric potential), 3.2 (properties), 3.3 (potentials from charge distributions), 3.4 (relation to the field), 3.5 (applications).

The four primers 00a to 00d taught the mathematics of chapter 1. This one teaches the physics that notebooks `02_superposition` and `03_potential_and_gradient` compute with: what charge is, the force between charges, the electric field as the thing that carries that force through space, superposition as the one rule that makes everything computable, and the potential as a scalar that stores the same information as the field in a form that is cheaper to compute and easier to read. Every result is derived by hand, evaluated with numbers, and then reproduced in code with `emlib`.

## 1. Warm-up

Try on paper first. Answers are in section 7.

1. Write Coulomb's law for the force on charge $q_2$ due to $q_1$ as a vector equation. Which way does the unit vector in it point, and what happens to the direction of the force when the charges have opposite signs?
2. Define the electric field. Why does the definition not depend on the size of the test charge used to measure it?
3. A dipole is two equal and opposite charges a small distance apart. Far away, how does $|\mathbf{E}|$ fall with distance, and why faster than for a single charge?
4. Write the field of a volume charge density $\rho(\mathbf{r}')$ as an integral. Which variable is integrated over, and which is held fixed?
5. Define the potential difference between two points as a line integral of $\mathbf{E}$. What property of the electrostatic field makes that definition possible?
6. Write $V$ for a point charge with $V = 0$ at infinity. What does $V$ look like far from a dipole?
7. State $\mathbf{E} = -\nabla V$. How are equipotential surfaces related to field lines, and where is the field strongest on a contour map of $V$?


## 2. The ideas

**Charge.** Matter carries a property called electric charge, measured in coulombs (C). It comes in two signs. Like signs repel, opposite signs attract, and the signs were chosen so that the total charge of a neutral atom is zero: the proton carries $+e$ and the electron $-e$ with $e = 1.602\times10^{-19}\,$C. Three facts about charge do most of the work in this course. It is conserved: the total charge in a closed region changes only when charge flows across its boundary, never by creation or destruction. It is quantised: every charge you will ever meet is a whole number of $e$, although for anything macroscopic the number is so large that we treat charge as a continuous fluid. And it is additive: the charge of a body is the sum of the charges of its parts. A coulomb is an enormous amount of charge. Rubbing a balloon transfers nanocoulombs; a lightning bolt carries a few coulombs; two charges of one coulomb each placed one metre apart would push on each other with a force of nine billion newtons.

**Force.** Coulomb's law gives the force between two point charges at rest. It is proportional to each charge, falls as the inverse square of the distance, and acts along the line joining them. That is the whole experimental input of electrostatics. Everything else in chapters 2 to 8 is mathematics applied to this one law plus one rule: superposition. Superposition says the force on a charge from many other charges is the vector sum of the forces from each one taken alone, as if the others were not there. There is no interference between pairs and no three-body term. That is what makes the subject computable: a problem with a million charges is a million applications of the same formula added up, which is exactly the kind of work a computer is good at.

**Field.** Coulomb's law describes action at a distance: charge $q_1$ pushes on $q_2$ with no visible connection. The electric field is the idea that replaces action at a distance with a local property of space. Each charge sets up a field $\mathbf{E}(\mathbf{r})$ at every point, whether or not anything is there to feel it, and any other charge $q$ placed at $\mathbf{r}$ feels the force $q\mathbf{E}(\mathbf{r})$. This splits every problem into two halves: the sources make the field, and the field acts on whatever is placed in it. The split is more than bookkeeping. The field carries energy and momentum, it takes time to change when a charge moves (chapter 17), and in the end it is the field, not the charges, that light is made of. For now the practical consequence is that we compute $\mathbf{E}$ once, as a vector field on a grid, and read off the force on anything we like afterwards.

**Continuous distributions.** Real charged bodies contain so many elementary charges that we describe them by densities: charge per length $\lambda$ on a wire, per area $\sigma$ on a surface, per volume $\rho$ in a body. The field of such a body is the superposition sum turned into an integral over the body. In notebook 00b every integral was a limit of a sum; here that is also the algorithm. Cut the body into small pieces, treat each piece as a point charge at its centre, add the Coulomb fields, and refine. Chapter 2 does the integrals by hand for a few shapes with enough symmetry (a ring, a line, a disk); the computer does them for any shape by brute force. Both are used below, and the hand results are the tests for the code.

**Work and potential.** Moving a charge through a field takes work, positive or negative. For the electrostatic field that work depends only on where you start and end, not on the path taken (the field is conservative, which notebooks 00b and 00c connected to zero curl). A path-independent work defines a potential energy, and potential energy per unit charge is the electric potential $V$. It is a scalar field, one number per point instead of three, and it contains the same information as $\mathbf{E}$: the field is minus the gradient of the potential. Computing $V$ first and differentiating is cheaper than computing $\mathbf{E}$ directly (a scalar sum with one power of distance less), it is easier to read (contour maps instead of arrow plots), and it is the quantity that conductors, batteries and voltmeters deal in. Notebook 03 is built on it.

**Energy.** Assembling charges costs work, and that work is stored in the configuration. Chapter 3 writes it as a sum over pairs of charges, or equivalently as half the sum of each charge times the potential the others make at its position. In module 2 the same energy is rewritten as an integral of $|\mathbf{E}|^2$ over all space, which says the energy lives in the field. That rewriting is the basis of the capacitance calculations in M2.4, and section 3.7 below sets it up.


## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import fields, grids
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=4, suppress=True)
e_charge, m_proton, m_electron, G_newton = 1.602176634e-19, 1.67262192e-27, 9.1093837e-31, 6.67430e-11
print(f"eps0 = {EPS0:.4e} C^2/(N m^2)     k = 1/(4 pi eps0) = {K:.4e} N m^2/C^2")

### 3.1 Charge and Coulomb's law (section 2.1)

Two point charges $q_1$ at $\mathbf{r}_1$ and $q_2$ at $\mathbf{r}_2$. The force on $q_2$ is

$$\mathbf{F}_{2} = \frac{1}{4\pi\epsilon_0}\,\frac{q_1 q_2}{|\mathbf{r}_2 - \mathbf{r}_1|^2}\,\hat{\mathbf{R}}, \qquad \hat{\mathbf{R}} = \frac{\mathbf{r}_2 - \mathbf{r}_1}{|\mathbf{r}_2 - \mathbf{r}_1|}.$$

Read it piece by piece. The unit vector $\hat{\mathbf{R}}$ points from the charge doing the pushing ($q_1$) to the charge being pushed ($q_2$). If $q_1q_2 > 0$ (like signs) the force on $q_2$ is along $\hat{\mathbf{R}}$, away from $q_1$: repulsion. If $q_1q_2 < 0$ the sign flips the force back toward $q_1$: attraction. The force on $q_1$ is equal and opposite, as Newton's third law requires. The magnitude falls as $1/R^2$: double the distance, a quarter of the force.

The constant $\epsilon_0 = 8.854\times10^{-12}\,\mathrm{C^2\,N^{-1}\,m^{-2}}$ is called the permittivity of free space. It is written in this odd way, with a $4\pi$, so that later formulas for spheres come out clean (Gauss' law in module 2 is where the $4\pi$ gets absorbed). The combination $k = 1/4\pi\epsilon_0 = 8.988\times10^9\,\mathrm{N\,m^2\,C^{-2}}$ is what you use for arithmetic. The code keeps both as `EPS0` and `K`.

A useful rewriting replaces the unit vector by the displacement over its cube, $\hat{\mathbf{R}}/R^2 = \mathbf{R}/R^3$, which avoids computing the unit vector separately. Notebook 00a, section 3.4, already used this form.

**Worked example 1.** $q_1 = +2\,$nC and $q_2 = -3\,$nC, $5\,$cm apart. Magnitude: $kq_1q_2/R^2 = 8.988\times10^9\times(2\times10^{-9})(3\times10^{-9})/(0.05)^2 = 8.988\times10^9\times6\times10^{-18}/2.5\times10^{-3} = 2.16\times10^{-5}\,$N, about $22\,\mu$N. The product $q_1q_2$ is negative, so the force on each charge points toward the other: attraction. Small in everyday terms, but the charges are tiny too. Scale both up to $1\,\mu$C: $8.988\times10^9\times10^{-12}/2.5\times10^{-3} = 3.6\,$N, the weight of a bag of sugar, from a millionth of a coulomb.

**Worked example 2 (superposition of forces).** $q_3 = +1\,$nC at the origin, $q_1 = +2\,$nC at $(3\,\mathrm{cm}, 0)$ and $q_2 = +2\,$nC at $(0, 4\,\mathrm{cm})$. Force from $q_1$: magnitude $8.988\times10^9\times2\times10^{-18}/(0.03)^2 = 2.00\times10^{-5}\,$N, direction from $q_1$ to $q_3$, which is $-\hat{\mathbf{x}}$. Force from $q_2$: $8.988\times10^9\times2\times10^{-18}/(0.04)^2 = 1.12\times10^{-5}\,$N along $-\hat{\mathbf{y}}$. Net force $(-20.0, -11.2)\,\mu$N, magnitude $\sqrt{20.0^2 + 11.2^2} = 22.9\,\mu$N, pointing into the third quadrant at $29^\circ$ below the $-x$ axis. Each pair was computed as if the third charge did not exist, then the vectors were added. That is superposition.

**Worked example 3 (how strong is it?).** Two protons: electric repulsion $ke^2/R^2$, gravitational attraction $Gm_p^2/R^2$. The $R^2$ cancels in the ratio:
$$\frac{F_e}{F_g} = \frac{ke^2}{Gm_p^2} = \frac{8.988\times10^9\times(1.602\times10^{-19})^2}{6.674\times10^{-11}\times(1.673\times10^{-27})^2} = 1.2\times10^{36}.$$
The electric force between elementary particles is stronger than gravity by thirty-six orders of magnitude, at every distance. Gravity dominates the large-scale world only because matter is so nearly neutral that the electric forces cancel. That near-perfect cancellation is also why a nanocoulomb of unbalanced charge is already a lot.


In [ ]:
def coulomb_force(q1, r1, q2, r2):
    '''Force on q2 due to q1 (vector).'''
    R = np.asarray(r2, float) - np.asarray(r1, float)
    return K * q1 * q2 * R / np.linalg.norm(R) ** 3

# Worked example 1
F = coulomb_force(2e-9, [0, 0, 0], -3e-9, [0.05, 0, 0])
print(f"force on q2 = {F * 1e6} uN   magnitude {np.linalg.norm(F):.3e} N   (negative x: pulled back toward q1, attraction)")

# Worked example 2
F1 = coulomb_force(2e-9, [0.03, 0, 0], 1e-9, [0, 0, 0])
F2 = coulomb_force(2e-9, [0, 0.04, 0], 1e-9, [0, 0, 0])
Fnet = F1 + F2
print(f"from q1: {F1 * 1e6} uN   from q2: {F2 * 1e6} uN   net: {Fnet * 1e6} uN   |net| = {np.linalg.norm(Fnet) * 1e6:.1f} uN"
      f"   angle = {np.degrees(np.arctan2(Fnet[1], Fnet[0])):.1f} deg")

# Worked example 3
print(f"electric / gravitational force between two protons: {K * e_charge**2 / (G_newton * m_proton**2):.2e}")

### 3.2 The electric field (section 2.2)

Place a small test charge $q$ at a point $\mathbf{r}$ and measure the force $\mathbf{F}$ on it. The electric field at that point is defined as the force per unit charge,

$$\mathbf{E}(\mathbf{r}) = \frac{\mathbf{F}}{q}.$$

Two things make this a good definition. First, by Coulomb's law every force on $q$ is proportional to $q$, so the ratio does not depend on which test charge you use: $\mathbf{E}$ is a property of the point, not of the probe. Second, the test charge must be small enough not to disturb the sources. A large $q$ would push the source charges around (if they are free to move, as on a conductor) and you would measure the field of a rearranged system. The definition is really a limit $q \to 0$, and in practice it means "small compared with the sources".

For a single point charge $q_1$ at $\mathbf{r}_1$, dividing Coulomb's law by $q_2$ gives
$$\mathbf{E}(\mathbf{r}) = \frac{1}{4\pi\epsilon_0}\,\frac{q_1}{|\mathbf{r} - \mathbf{r}_1|^2}\,\hat{\mathbf{R}} = \frac{q_1}{4\pi\epsilon_0}\frac{\mathbf{r} - \mathbf{r}_1}{|\mathbf{r} - \mathbf{r}_1|^3}.$$
It points away from a positive charge and toward a negative one, and falls as $1/R^2$. The force on any charge $q$ placed at $\mathbf{r}$ is then $q\mathbf{E}(\mathbf{r})$, along $\mathbf{E}$ for positive $q$ and against it for negative $q$.

Units: newtons per coulomb, N/C. Section 3.5 will show that this is the same as volts per metre, V/m, which is the unit everyone uses. Typical sizes: a few hundred V/m under a fair-weather sky, $10^6\,$V/m before air breaks down in a spark, $10^{11}\,$V/m inside an atom.

**Worked example 4.** $q_1 = +5\,$nC at the origin. Field at $10\,$cm: $E = kq_1/R^2 = 8.988\times10^9\times5\times10^{-9}/0.01 = 4494\,$V/m, pointing away from the charge. At $20\,$cm: $4494/4 = 1123\,$V/m. An electron placed at $10\,$cm feels a force $eE = 1.602\times10^{-19}\times4494 = 7.2\times10^{-16}\,$N, directed toward the charge because the electron is negative, and accelerates at $F/m_e = 7.2\times10^{-16}/9.11\times10^{-31} = 7.9\times10^{14}\,\mathrm{m/s^2}$. Fields that are modest in everyday units are violent on the scale of an electron.

**Worked example 5 (the same point, a different probe).** Replace the electron by a proton at the same spot: force $7.2\times10^{-16}\,$N, now pointing away from the charge, acceleration $1836$ times smaller. The field is identical in both cases, $4494\,$V/m outward. Only the force and the acceleration changed, because they belong to the probe.


In [ ]:
src, q = np.array([[0.0, 0.0, 0.0]]), np.array([5e-9])
pts = np.array([[0.10, 0, 0], [0.20, 0, 0], [0, 0.10, 0], [0.06, 0.08, 0]])
E = fields.point_charges_E(src, q, pts)
for p, e in zip(pts, E):
    print(f"at {p} (r = {np.linalg.norm(p):.2f} m):  E = {e} V/m   |E| = {np.linalg.norm(e):.1f} V/m")

E10 = np.linalg.norm(E[0])
for name, charge, mass in (("electron", -e_charge, m_electron), ("proton", e_charge, m_proton)):
    F = charge * E[0]
    print(f"{name:8s} at 10 cm: force {F * 1e15} fN  (|F| = {np.linalg.norm(F):.2e} N),  acceleration {np.linalg.norm(F) / mass:.2e} m/s^2")

### 3.3 Superposition and field lines (sections 2.2, 2.3)

For $N$ point charges the field is the vector sum of the single-charge fields:
$$\mathbf{E}(\mathbf{r}) = \frac{1}{4\pi\epsilon_0}\sum_{i=1}^{N} q_i\,\frac{\mathbf{r} - \mathbf{r}_i}{|\mathbf{r} - \mathbf{r}_i|^3}.$$
Each term is evaluated as if the other charges did not exist. This is the formula that `emlib.fields.point_charges_E` implements: for every observation point and every source, form the displacement, divide by its cube, weight by the charge, add.

Pictures of fields follow the two conventions of notebook 00a. Arrow plots show $\mathbf{E}$ at a sample of points. Field lines are curves that follow $\mathbf{E}$ everywhere; for electrostatic fields they obey four rules that are worth memorising because they let you sketch a field without computing it. Lines start on positive charges and end on negative charges (or at infinity). The number of lines leaving a charge is proportional to the charge. Lines never cross, because $\mathbf{E}$ has one direction at each point. And where lines crowd together the field is strong: the density of lines is proportional to $|\mathbf{E}|$, a fact that Gauss' law turns into a theorem in module 2.

**The dipole.** The most important two-charge system is $+q$ and $-q$ separated by a small displacement $\mathbf{d}$ pointing from the negative to the positive charge. Its dipole moment is $\mathbf{p} = q\mathbf{d}$. Molecules such as water are dipoles, and chapter 5 builds dielectrics out of them. Two directions have simple closed forms. Put $+q$ at $(-a, 0)$ and $-q$ at $(a, 0)$, so $d = 2a$ and $\mathbf{p} = -2aq\,\hat{\mathbf{x}}$.

**Worked example 6 (field on the perpendicular bisector).** At $(0, y)$ each charge is at distance $\sqrt{a^2 + y^2}$. The field of $+q$ points from $(-a, 0)$ toward the observation point, along $(a, y)/\sqrt{a^2 + y^2}$. The field of $-q$ points from the observation point toward $(a, 0)$, along $(a, -y)/\sqrt{a^2 + y^2}$. Both have magnitude $kq/(a^2 + y^2)$. The $y$ components cancel, the $x$ components add:
$$\mathbf{E} = \frac{2kqa}{(a^2 + y^2)^{3/2}}\,\hat{\mathbf{x}} = \frac{kp}{(a^2 + y^2)^{3/2}}\,\hat{\mathbf{x}} \;\longrightarrow\; \frac{kp}{y^3}\,\hat{\mathbf{x}} \quad (y \gg a).$$
Since $\mathbf{p}$ points along $-\hat{\mathbf{x}}$, the field on the bisector is antiparallel to $\mathbf{p}$, with magnitude $kp/r^3$ far away. With $q = 1\,$nC and $a = 1\,$cm ($p = 2\times10^{-11}\,$C m) at $y = 10\,$cm: exact $2\times8.988\times10^9\times10^{-9}\times0.01/(0.0101)^{3/2} = 0.1798/1.015\times10^{-3} = 177.1\,$V/m; far-field formula $kp/y^3 = 179.8\,$V/m, $1.5\%$ high at ten times the half-separation.

**Worked example 7 (field on the axis).** At $(x, 0)$ with $x > a$: $+q$ contributes $kq/(x + a)^2$ along $+\hat{\mathbf{x}}$, $-q$ contributes $kq/(x - a)^2$ along $-\hat{\mathbf{x}}$ (toward the negative charge). Net
$$E_x = kq\left[\frac{1}{(x + a)^2} - \frac{1}{(x - a)^2}\right] = \frac{-4kqax}{(x^2 - a^2)^2} \;\longrightarrow\; -\frac{4kqa}{x^3} = -\frac{2kp}{x^3} \quad (x \gg a).$$
On the axis the field is parallel to $\mathbf{p}$ (here $-\hat{\mathbf{x}}$) and twice as strong as on the bisector at the same distance. At $x = 10\,$cm: exact $-8.988\times(0.004)/(0.0099)^2 = -366.8\,$V/m, far-field $-359.5\,$V/m.

Both results fall as $1/r^3$, one power faster than a point charge. The reason is cancellation: from far away the two charges almost coincide and their fields almost cancel, and what survives is the small difference between two $1/r^2$ fields displaced by $d$, which is of order $d/r$ times $1/r^2$. The same reasoning says a neutral quadrupole (two opposite dipoles) falls as $1/r^4$, and so on. The far field of any charge distribution is dominated by its lowest non-vanishing moment: net charge if there is one, dipole moment if not.


In [ ]:
a, qd = 0.01, 1e-9
dip_src = np.array([[-a, 0, 0], [a, 0, 0]]); dip_q = np.array([qd, -qd])
p_mag = qd * 2 * a
ys = np.array([0.03, 0.05, 0.10, 0.30])
E_bis = fields.point_charges_E(dip_src, dip_q, np.stack([0 * ys, ys, 0 * ys], axis=-1))
E_ax = fields.point_charges_E(dip_src, dip_q, np.stack([ys, 0 * ys, 0 * ys], axis=-1))
print("    r      bisector E_x    exact kp/(a^2+r^2)^1.5    far kp/r^3   |   axis E_x    exact      far -2kp/r^3")
for r, eb, ea in zip(ys, E_bis, E_ax):
    print(f"{r:5.2f}   {eb[0]:12.2f}   {K * p_mag / (a**2 + r**2)**1.5:18.2f}   {K * p_mag / r**3:12.2f}   |"
          f"   {ea[0]:9.2f}   {-4 * K * qd * a * r / (r**2 - a**2)**2:9.2f}   {-2 * K * p_mag / r**3:9.2f}")
print(f"the far-field formulas are within 2% once r > 10a; the E_y components on both lines vanish by symmetry: {np.abs(E_bis[:, 1]).max():.1e}, {np.abs(E_ax[:, 1]).max():.1e}")

# Field lines of the dipole and of two like charges, with arrows for comparison.
x, y, P = grids.grid2d((-0.05, 0.05), (-0.05, 0.05), 81)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
for ax, charges, name in zip(axes, ((qd, -qd), (qd, qd)), ("dipole: lines run from + to -", "two like charges: lines repel each other")):
    E = fields.point_charges_E(dip_src, np.array(charges), P)
    Ex, Ey = E[..., 0], E[..., 1]
    ax.streamplot(x, y, Ex.T, Ey.T, color=np.log10(np.hypot(Ex, Ey)).T, cmap="plasma", density=1.4, linewidth=0.8)
    for (sx, sy, _), c, col in zip(dip_src, charges, ("C3", "C0")):
        ax.plot(sx, sy, "o", color="C3" if c > 0 else "C0", ms=10)
    ax.set_aspect("equal"); ax.set_title(name)
plt.show()

### 3.4 Continuous distributions (section 2.4)

A charged body contains some $10^{20}$ elementary charges. Instead of listing them we describe the body by a density, and the sum over charges becomes an integral over the body. Three cases:

| body | density | small charge | field |
|---|---|---|---|
| wire or curve | $\lambda$ (C/m) | $dq = \lambda\,dl$ | $\mathbf{E}(\mathbf{r}) = k\int \lambda(\mathbf{r}')\,\dfrac{\mathbf{r} - \mathbf{r}'}{|\mathbf{r} - \mathbf{r}'|^3}\,dl'$ |
| surface | $\sigma$ (C/m$^2$) | $dq = \sigma\,dS$ | same with $\sigma\,dS'$ |
| volume | $\rho$ (C/m$^3$) | $dq = \rho\,dv$ | same with $\rho\,dv'$ |

The primes mark the source point $\mathbf{r}'$, which is the variable of integration, running over the body. The field point $\mathbf{r}$ is held fixed during the integral. Confusing the two is the commonest mistake in this chapter. The integrand is just Coulomb's law for the small charge $dq$ at $\mathbf{r}'$, and the integral is superposition.

Two ways to evaluate the integral. By hand, when the geometry has enough symmetry that most components cancel and the remaining one-dimensional integral is tractable. By computer, always: cut the body into $N$ pieces, assign each piece the charge $dq$ at its centre, and call the point-charge sum. The error comes from treating each piece as a point and falls as the pieces shrink, provided the field point is not inside a piece (notebook 02 studies that convergence). The two methods check each other, and below each hand result becomes a test for the code.

**Worked example 8 (a ring, on its axis).** A ring of radius $R$ in the $xy$ plane carries charge $Q$ uniformly, so $\lambda = Q/2\pi R$. Field point on the axis at $(0, 0, z)$. A piece $dq$ on the ring is at distance $\sqrt{R^2 + z^2}$ from the field point, so its field has magnitude $k\,dq/(R^2 + z^2)$ and points from the piece to the field point. Decompose that direction into a part along $\hat{\mathbf{z}}$, with cosine $z/\sqrt{R^2 + z^2}$, and a part in the $xy$ plane. The in-plane parts cancel: the piece diametrically opposite gives the mirror-image contribution. The $z$ parts all add:
$$E_z = \int \frac{k\,dq}{R^2 + z^2}\cdot\frac{z}{\sqrt{R^2 + z^2}} = \frac{kz}{(R^2 + z^2)^{3/2}}\int dq = \frac{kQz}{(R^2 + z^2)^{3/2}}.$$
Checks: at $z = 0$ the field is zero (by symmetry every piece is cancelled by its opposite); for $z \gg R$ it becomes $kQ/z^2$, a point charge; it is largest at $z = R/\sqrt2$. With $Q = 10\,$nC, $R = 5\,$cm, $z = 5\,$cm: $8.988\times10^9\times10^{-8}\times0.05/(0.005)^{3/2} = 4.494/3.536\times10^{-4} = 12\,710\,$V/m.

**Worked example 9 (a finite line, on its perpendicular bisector).** A straight segment from $x = -L$ to $x = L$ carries $\lambda$ per unit length. Field point at $(0, d)$. The piece $\lambda\,dx'$ at $x'$ is at distance $\sqrt{x'^2 + d^2}$; its field has magnitude $k\lambda\,dx'/(x'^2 + d^2)$ and makes an angle with the $y$ axis whose cosine is $d/\sqrt{x'^2 + d^2}$. The $x$ components cancel between $x'$ and $-x'$. The $y$ component:
$$E_y = k\lambda d\int_{-L}^{L}\frac{dx'}{(x'^2 + d^2)^{3/2}} = k\lambda d\left[\frac{x'}{d^2\sqrt{x'^2 + d^2}}\right]_{-L}^{L} = \frac{2k\lambda L}{d\sqrt{L^2 + d^2}} = \frac{kQ}{d\sqrt{L^2 + d^2}},$$
with $Q = 2\lambda L$. (The antiderivative can be checked by differentiating it.) Two limits. Far away, $d \gg L$: $kQ/d^2$, a point charge. Close in or for a long line, $L \gg d$:
$$E_y \to \frac{2k\lambda}{d} = \frac{\lambda}{2\pi\epsilon_0 d}.$$
This is the field of an infinite line charge, falling as $1/d$ rather than $1/d^2$ because the far parts of the line keep contributing. Module 2 gets the same result from Gauss' law in two lines. With $\lambda = 20\,$nC/m, $L = 0.5\,$m, $d = 0.1\,$m: exact $2\times8.988\times10^9\times2\times10^{-8}\times0.5/(0.1\times\sqrt{0.26}) = 179.8/0.05099 = 3525\,$V/m; infinite-line formula $3595\,$V/m, $2\%$ high because $L$ is only five times $d$.


In [ ]:
# Worked example 8: the ring, as N point charges on the circle.
Q_ring, R_ring, N = 10e-9, 0.05, 400
phi = 2 * np.pi * (np.arange(N) + 0.5) / N
ring_src = np.stack([R_ring * np.cos(phi), R_ring * np.sin(phi), np.zeros(N)], axis=-1)
ring_q = np.full(N, Q_ring / N)
zs = np.array([0.0, 0.02, R_ring / np.sqrt(2), 0.05, 0.10, 0.50])
E_ring = fields.point_charges_E(ring_src, ring_q, np.stack([0 * zs, 0 * zs, zs], axis=-1))
E_ring_exact = K * Q_ring * zs / (R_ring**2 + zs**2) ** 1.5
print("ring on axis:   z [m]   E_z sum [V/m]   kQz/(R^2+z^2)^1.5   point charge kQ/z^2")
for z, e, ex in zip(zs, E_ring[:, 2], E_ring_exact):
    print(f"              {z:6.4f}   {e:13.1f}   {ex:18.1f}   {K * Q_ring / z**2 if z > 0 else np.inf:16.1f}")
print(f"transverse components on the axis (should vanish by symmetry): {np.abs(E_ring[:, :2]).max():.1e}")

In [ ]:
# Worked example 9: the finite line, and its approach to the infinite-line law.
lam, L_half, N = 20e-9, 0.5, 2000
xs = -L_half + (np.arange(N) + 0.5) * (2 * L_half / N)
line_src = np.stack([xs, 0 * xs, 0 * xs], axis=-1); line_q = np.full(N, lam * 2 * L_half / N)
ds = np.array([0.01, 0.03, 0.10, 0.30, 1.0, 3.0])
E_line = fields.point_charges_E(line_src, line_q, np.stack([0 * ds, ds, 0 * ds], axis=-1))[:, 1]
print("finite line:  d [m]   E_y sum    2 k lam L / (d sqrt(L^2+d^2))    infinite line lam/(2 pi eps0 d)    point charge kQ/d^2")
for d, e in zip(ds, E_line):
    exact = 2 * K * lam * L_half / (d * np.sqrt(L_half**2 + d**2))
    print(f"             {d:5.2f}   {e:9.1f}   {exact:28.1f}   {lam / (2 * np.pi * EPS0 * d):30.1f}   {K * lam * 2 * L_half / d**2:18.1f}")
print("close to the line (d << L) the infinite-line law holds; far away (d >> L) the point-charge law takes over")

### 3.5 Work, conservative fields and the potential (sections 3.1 to 3.3)

Push a charge $q$ along a path from $\mathbf{a}$ to $\mathbf{b}$ through a field $\mathbf{E}$. The field does work $W = \int_{\mathbf{a}}^{\mathbf{b}} q\mathbf{E}\cdot d\mathbf{l}$ on it (notebook 00b, section 3.3). For the electrostatic field this work is the same for every path between the same two endpoints. Three equivalent ways to say that: the integral around any closed loop is zero, the curl of $\mathbf{E}$ is zero, and $\mathbf{E}$ is the gradient of some scalar. Notebook 00d, section 3.6, showed that the last follows from the first two. The physical reason is that the field of a point charge is radial and depends on distance only, so the work done moving against it depends only on the change in distance; and superposition carries the property over to every electrostatic field.

A path-independent work defines a potential energy: $U(\mathbf{b}) - U(\mathbf{a}) = -W$, the work you must do against the field. Dividing by the charge gives a quantity that belongs to the field alone, the electric potential:

$$V(\mathbf{b}) - V(\mathbf{a}) = -\int_{\mathbf{a}}^{\mathbf{b}}\mathbf{E}\cdot d\mathbf{l}, \qquad U = qV.$$

The sign deserves a sentence. Moving along $\mathbf{E}$ (the direction a positive charge is pushed) the integral is positive, so $V$ decreases: the field points downhill in potential, from high $V$ to low $V$. A positive charge released from rest moves toward lower potential and lower potential energy, as a ball rolls downhill. A negative charge moves toward higher $V$, because its energy $qV$ is lower there.

Only differences of $V$ are defined by the integral, so we choose a reference point and declare $V = 0$ there. For any finite collection of charges the convenient choice is infinity, far from everything. Then the potential of a point charge $q_1$ at distance $r$ is the work per unit charge to bring a test charge in from infinity:
$$V(r) = -\int_{\infty}^{r}\frac{kq_1}{r'^2}\,dr' = \left[\frac{kq_1}{r'}\right]_{\infty}^{r} = \frac{kq_1}{r} = \frac{q_1}{4\pi\epsilon_0 r}.$$
(Integrate along a radial line; the path does not matter.) Positive near a positive charge, falling as $1/r$: one power slower than the field.

Units. $V$ is joules per coulomb, which is the volt (V). From $V = -\int\mathbf{E}\cdot d\mathbf{l}$, the field is volts per metre, and this is why N/C and V/m are the same unit. A convenient unit of energy at the atomic scale is the electron volt, the energy an electron gains falling through one volt: $1\,\mathrm{eV} = 1.602\times10^{-19}\,$J.

Superposition carries over, and the sum is now a scalar sum, no components, no cross terms:
$$V(\mathbf{r}) = \frac{1}{4\pi\epsilon_0}\sum_i\frac{q_i}{|\mathbf{r} - \mathbf{r}_i|}, \qquad V(\mathbf{r}) = \frac{1}{4\pi\epsilon_0}\int\frac{\rho(\mathbf{r}')\,dv'}{|\mathbf{r} - \mathbf{r}'|}.$$
This is why notebook 03 computes $V$ instead of $\mathbf{E}$: a third of the arithmetic, and the hand integrals are easier because there are no directions to keep track of.

**Worked example 10 (potential of a point charge, with numbers).** $q_1 = +5\,$nC. At $10\,$cm, $V = 8.988\times10^9\times5\times10^{-9}/0.1 = 449.4\,$V; at $20\,$cm, $224.7\,$V. To carry a charge $q = +2\,$nC from $20\,$cm in to $10\,$cm you must do work $q\,\Delta V = 2\times10^{-9}\times224.7 = 4.49\times10^{-7}\,$J against the field, along any path. Released at $10\,$cm it would gain that much kinetic energy by the time it reached $20\,$cm. An electron making the same inward trip has its energy lowered by $224.7\,$eV, which is $3.6\times10^{-17}\,$J.

**Worked example 11 (the ring again, now the potential).** Every piece of the ring is at the same distance $\sqrt{R^2 + z^2}$ from an axis point, so the scalar sum is immediate:
$$V(z) = k\int\frac{dq}{\sqrt{R^2 + z^2}} = \frac{kQ}{\sqrt{R^2 + z^2}}.$$
Compare the effort with worked example 8. And differentiating recovers the field: $E_z = -dV/dz = kQz/(R^2 + z^2)^{3/2}$, exactly worked example 8. With $Q = 10\,$nC, $R = 5\,$cm, $z = 5\,$cm: $V = 89.88/0.07071 = 1271\,$V.


In [ ]:
# Worked example 10
pts = np.array([[0.10, 0, 0], [0.20, 0, 0], [0.0, 0.0, 0.10], [0.06, 0.08, 0.0]])
V = fields.point_charges_V(src, q, pts)
print("potential of +5 nC at", pts[:, 0], "...:", V, "V")
print(f"work to move +2 nC from 20 cm to 10 cm: q (V(10) - V(20)) = {2e-9 * (V[0] - V[1]):.3e} J")
print(f"electron energy change on the same trip: {-(V[0] - V[1]):.1f} eV = {-(V[0] - V[1]) * e_charge:.2e} J")

# The same potential difference by direct line integration of E along three different paths.
E_func = lambda pts: fields.point_charges_E(src, q, pts)
t = np.linspace(0, 1, 4001)[:, None]
a_pt, b_pt = np.array([0.20, 0, 0]), np.array([0.10, 0, 0])
paths = {"straight":    a_pt + t * (b_pt - a_pt),
         "detour up":   np.concatenate([a_pt + t * np.array([0, 0.15, 0]), a_pt + np.array([0, 0.15, 0]) + t * (b_pt - a_pt), b_pt + np.array([0, 0.15, 0]) - t * np.array([0, 0.15, 0])]),
         "half circle": np.stack([0.15 + 0.05 * np.cos(np.pi * t[:, 0]), 0.05 * np.sin(np.pi * t[:, 0]), 0 * t[:, 0]], axis=-1)}
for name, path in paths.items():
    print(f"-integral of E.dl along the {name:11s} path: {-fields.line_integral(E_func, path):.3f} V   (V(b) - V(a) = {V[0] - V[1]:.3f} V)")

# Worked example 11
V_ring = fields.point_charges_V(ring_src, ring_q, np.stack([0 * zs, 0 * zs, zs], axis=-1))
print("\nring on axis: V sum =", V_ring, "\n              kQ/sqrt(R^2+z^2) =", K * Q_ring / np.sqrt(R_ring**2 + zs**2))
zf = np.linspace(0, 0.3, 3001); Vf = fields.point_charges_V(ring_src, ring_q, np.stack([0 * zf, 0 * zf, zf], axis=-1))
Ez_from_V = -np.gradient(Vf, zf[1] - zf[0])
print(f"-dV/dz at z = 5 cm: {np.interp(0.05, zf, Ez_from_V):.1f} V/m   worked example 8 gave 12710 V/m")

### 3.6 From the potential back to the field (section 3.4)

Take the definition $dV = -\mathbf{E}\cdot d\mathbf{l}$ for an infinitesimal step and compare with the gradient from notebook 00c, $dV = \nabla V\cdot d\mathbf{l}$. They agree for every step only if

$$\mathbf{E} = -\nabla V.$$

So the field is the downhill slope of the potential: it points in the direction of steepest decrease of $V$, with magnitude equal to that slope. Everything in notebook 00c about gradients now applies. Surfaces of constant $V$ (equipotentials) are perpendicular to $\mathbf{E}$, so field lines cross equipotentials at right angles. Where equipotentials are closely spaced the field is strong. And since a conductor in equilibrium has no field inside it (module 2), its whole surface is one equipotential, which is why field lines leave a conductor at right angles.

On a grid this is the algorithm of notebook 03: compute $V$ at every grid point by the scalar sum, then take central differences, $\mathbf{E} \approx -\nabla V$. The error is of order $h^2$ and the only expensive step is the sum. The direct superposition field computed by `point_charges_E` provides the check.

**Worked example 12 (the dipole potential and field).** $+q$ at $(0, 0, a)$ and $-q$ at $(0, 0, -a)$, so $\mathbf{p} = 2aq\,\hat{\mathbf{z}}$. At a point at distance $r$ and polar angle $\theta$ from the $z$ axis, the distances to the two charges are $r_\pm = \sqrt{r^2 \mp 2ar\cos\theta + a^2}$. For $r \gg a$ expand: $r_\pm \approx r \mp a\cos\theta$, so $1/r_\pm \approx (1/r)(1 \pm (a/r)\cos\theta)$ and
$$V = kq\left(\frac{1}{r_+} - \frac{1}{r_-}\right) \approx \frac{kq\cdot2a\cos\theta}{r^2} = \frac{k\,p\cos\theta}{r^2} = \frac{k\,\mathbf{p}\cdot\hat{\mathbf{r}}}{r^2}.$$
Falls as $1/r^2$, one power faster than a point charge, zero on the equatorial plane, positive on the $+q$ side. Now the field, using the spherical gradient from notebook 00d: $E_r = -\partial V/\partial r = 2kp\cos\theta/r^3$ and $E_\theta = -(1/r)\partial V/\partial\theta = kp\sin\theta/r^3$. On the axis ($\theta = 0$) $E_r = 2kp/r^3$ along $\mathbf{p}$; on the bisector ($\theta = 90^\circ$) only $E_\theta = kp/r^3$ survives, pointing along $-\mathbf{p}$. Those are worked examples 7 and 6, obtained here from one scalar function and two derivatives instead of two vector additions. With $p = 2\times10^{-11}\,$C m at $r = 10\,$cm, $\theta = 60^\circ$ the far-field formulas give $V = 8.988\times10^9\times2\times10^{-11}\times0.5/0.01 = 8.99\,$V, $E_r = 179.8\,$V/m, $E_\theta = 155.7\,$V/m. The code evaluates the exact two-charge sum at the same point: $V = 8.91\,$V, $E_r = 176.6\,$V/m, $E_\theta = 156.2\,$V/m, within $2\%$ of the far-field values because $r$ is ten times $a$. The terms dropped in the expansion are of relative size $(a/r)^2$.


In [ ]:
# V of the dipole on a 2D grid in the xz plane, then E = -grad V by central differences, compared with the direct sum.
az = 0.01
dz_src = np.array([[0, 0, az], [0, 0, -az]]); dz_q = np.array([qd, -qd]); p_vec = 2 * az * qd
x, z, P = grids.grid2d((-0.08, 0.08), (-0.08, 0.08), 161)       # P[..., 1] is y = 0; we use axes x and z
P = np.stack([P[..., 0], np.zeros_like(P[..., 0]), P[..., 1]], axis=-1)
h = grids.spacing(x, z)
V_grid = fields.point_charges_V(dz_src, dz_q, P)
E_from_V = -fields.gradient(V_grid, *h)                           # (nx, nz, 2): components along x and z
E_direct = fields.point_charges_E(dz_src, dz_q, P)[..., [0, 2]]
r_grid = np.hypot(P[..., 0], P[..., 2])
far = r_grid > 0.03
rel = np.linalg.norm(E_from_V - E_direct, axis=-1)[far] / np.linalg.norm(E_direct, axis=-1)[far]
print(f"-grad V versus direct superposition, r > 3 cm: max relative difference {rel.max():.1e}  (central-difference error, h = {h[0]:.4f} m)")

# Worked example 12 at r = 10 cm, theta = 60 degrees.
r0, th0 = 0.10, np.radians(60)
pt = np.array([[r0 * np.sin(th0), 0, r0 * np.cos(th0)]])
V_pt = fields.point_charges_V(dz_src, dz_q, pt)[0]; E_pt = fields.point_charges_E(dz_src, dz_q, pt)[0]
r_hat = pt[0] / r0; th_hat = np.array([np.cos(th0), 0, -np.sin(th0)])
print(f"dipole at r = 10 cm, theta = 60 deg:  V = {V_pt:.3f} V (far-field kp cos/r^2 = {K * p_vec * np.cos(th0) / r0**2:.3f})"
      f"   E_r = {E_pt @ r_hat:.1f} ({2 * K * p_vec * np.cos(th0) / r0**3:.1f})   E_theta = {E_pt @ th_hat:.1f} ({K * p_vec * np.sin(th0) / r0**3:.1f})")

fig, ax = plt.subplots(figsize=(6.5, 6))
levels = np.linspace(-60, 60, 25)
cs = ax.contour(x, z, V_grid.T, levels=levels, cmap="coolwarm"); ax.clabel(cs, fontsize=6, fmt="%.0f")
st = 8
Em = np.linalg.norm(E_from_V, axis=-1)
ax.quiver(P[::st, ::st, 0], P[::st, ::st, 2], (E_from_V[..., 0] / Em)[::st, ::st], (E_from_V[..., 1] / Em)[::st, ::st], color="k", scale=30, width=0.004)
ax.set_aspect("equal"); ax.set_xlabel("x [m]"); ax.set_ylabel("z [m]"); ax.set_title("dipole: equipotentials (V) and unit field arrows, always perpendicular"); plt.show()

### 3.7 Energy of a configuration of charges (section 3.5)

How much work does it take to assemble a set of point charges, bringing them in one at a time from infinity? The first charge costs nothing: there is no field yet. The second is brought in against the potential of the first, costing $q_2V_1(\mathbf{r}_2) = kq_1q_2/r_{12}$. The third costs $q_3[V_1(\mathbf{r}_3) + V_2(\mathbf{r}_3)]$, and so on. The total is a sum over every pair once,
$$W = k\sum_{\text{pairs } i<j}\frac{q_iq_j}{r_{ij}} = \frac12\sum_i q_iV_i,$$
where $V_i$ is the potential at charge $i$ due to all the others. The $\tfrac12$ corrects for each pair appearing twice in the double sum. This work is stored as potential energy in the configuration: release the charges and it comes back as kinetic energy. Negative $W$ means the configuration is bound, and work is needed to pull it apart.

**Worked example 13.** $q_1 = q_2 = +2\,$nC and $q_3 = -3\,$nC at the corners of an equilateral triangle of side $3\,$cm. Three pairs, all at distance $0.03\,$m: $q_1q_2 = 4\times10^{-18}$, $q_1q_3 = q_2q_3 = -6\times10^{-18}$, sum $-8\times10^{-18}\,\mathrm{C^2}$. $W = 8.988\times10^9\times(-8\times10^{-18})/0.03 = -2.40\times10^{-6}\,$J. Bound: the two attractions outweigh the one repulsion, and $2.4\,\mu$J of work is needed to separate the three charges to infinity. The code evaluates the pair sum and the $\tfrac12\sum q_iV_i$ form and shows they agree.

Where this leads. For a continuous distribution the sum becomes $W = \tfrac12\int\rho V\,dv$, and module 2 (notebook M2.4, chapter 8) rewrites it with Gauss' law and an integration by parts as
$$W = \frac{\epsilon_0}{2}\int|\mathbf{E}|^2\,dv,$$
an integral over all space of an energy density $\tfrac12\epsilon_0E^2$ that lives wherever there is field, charge or no charge. That form is what lets a grid solver compute the capacitance of any shape: solve for $V$, take the gradient, integrate the energy density.


In [ ]:
side = 0.03
tri_src = side * np.array([[0, 0, 0], [1, 0, 0], [0.5, np.sqrt(3) / 2, 0]])
tri_q = np.array([2e-9, 2e-9, -3e-9])
W_pairs = sum(K * tri_q[i] * tri_q[j] / np.linalg.norm(tri_src[i] - tri_src[j]) for i in range(3) for j in range(i + 1, 3))
V_at_each = fields.point_charges_V(tri_src, tri_q, tri_src)     # the self term is excluded by the library (zero at coincidence)
W_half = 0.5 * np.sum(tri_q * V_at_each)
print(f"pair sum: {W_pairs:.4e} J     (1/2) sum q_i V_i: {W_half:.4e} J     by hand: -2.40e-06 J")
print("potential at each charge due to the other two:", V_at_each, "V")

### Exercise: the potential of a finite line

Replace the body of `line_V_bisector` so that it returns the potential at distance $d$ from the centre of a straight segment of length $2L$ and total charge $Q$, on its perpendicular bisector, by evaluating the scalar integral $k\int_{-L}^{L}\lambda\,dx'/\sqrt{x'^2 + d^2}$ in closed form. The antiderivative of $1/\sqrt{x^2 + d^2}$ is $\ln\big(x + \sqrt{x^2 + d^2}\big)$, which gives
$$V(d) = \frac{k\lambda}{1}\ln\frac{L + \sqrt{L^2 + d^2}}{-L + \sqrt{L^2 + d^2}}, \qquad \lambda = \frac{Q}{2L}.$$
The check compares with a point-charge sum at several distances and also verifies that $-dV/dd$ reproduces the field of worked example 9.


In [ ]:
def line_V_bisector(d, L, Q):
    '''Potential on the perpendicular bisector of a line of length 2L and charge Q, at distance d from its centre.'''
    lam = Q / (2 * L)                                                  # replace from here with your own version
    return K * lam * np.log((L + np.sqrt(L**2 + d**2)) / (-L + np.sqrt(L**2 + d**2)))


Q_line = lam * 2 * L_half
d_test = np.array([0.02, 0.05, 0.1, 0.3, 1.0])
V_hand = line_V_bisector(d_test, L_half, Q_line)
V_sum = fields.point_charges_V(line_src, line_q, np.stack([0 * d_test, d_test, 0 * d_test], axis=-1))
print("d:        ", d_test); print("closed form:", V_hand); print("point sum:  ", V_sum)
assert np.allclose(V_hand, V_sum, rtol=1e-3)
eps = 1e-6
E_from_V = -(line_V_bisector(d_test + eps, L_half, Q_line) - line_V_bisector(d_test - eps, L_half, Q_line)) / (2 * eps)
E_direct = 2 * K * lam * L_half / (d_test * np.sqrt(L_half**2 + d_test**2))
print("-dV/dd:    ", E_from_V); print("worked ex 9:", E_direct)
assert np.allclose(E_from_V, E_direct, rtol=1e-5)
print("exercise passed")

## 4. Verify

Two closed-form checks on an arbitrary cluster of charges, where nothing is symmetric and no hand formula exists for the field itself.

1. Path independence. The line integral of the superposition field between two points along two very different paths must agree, and $-\int\mathbf{E}\cdot d\mathbf{l}$ must equal $V(\mathbf{b}) - V(\mathbf{a})$ computed from the scalar sum. This is the statement that the electrostatic field is conservative, tested on a field computed without ever assuming it.
2. $\mathbf{E} = -\nabla V$ on a 3D grid, compared with the direct superposition field away from the charges. The difference is the central-difference error and should be small where the field varies slowly.


In [ ]:
rng = np.random.default_rng(3)
cl_src = rng.uniform(-0.02, 0.02, size=(6, 3)); cl_q = rng.uniform(-3e-9, 3e-9, size=6)
E_cl = lambda pts: fields.point_charges_E(cl_src, cl_q, pts)
a_pt, b_pt = np.array([0.06, -0.05, 0.03]), np.array([-0.05, 0.06, -0.04])
t = np.linspace(0, 1, 8001)[:, None]
straight = a_pt + t * (b_pt - a_pt)
bulge = straight + 0.08 * np.sin(np.pi * t) * np.array([1.0, 1.0, 0.5])      # same endpoints, a wide detour
I1, I2 = fields.line_integral(E_cl, straight), fields.line_integral(E_cl, bulge)
dV = (fields.point_charges_V(cl_src, cl_q, b_pt[None])[0] - fields.point_charges_V(cl_src, cl_q, a_pt[None])[0])
print(f"-integral along straight path: {-I1:.4f} V    along the detour: {-I2:.4f} V    V(b) - V(a) from the scalar sum: {dV:.4f} V")
assert abs(I1 - I2) < 1e-3 * abs(dV) and abs(-I1 - dV) < 1e-3 * abs(dV)

x3, y3, z3, P3 = grids.grid3d((-0.08, 0.08), (-0.08, 0.08), (-0.08, 0.08), 65)
h3 = grids.spacing(x3, y3, z3)
V3 = fields.point_charges_V(cl_src, cl_q, P3)
E3_from_V = -fields.gradient(V3, *h3)
E3_direct = fields.point_charges_E(cl_src, cl_q, P3)
far = (grids.magnitude(P3) > 0.045) & (grids.magnitude(P3) < 0.075)
rel3 = (np.linalg.norm(E3_from_V - E3_direct, axis=-1) / np.linalg.norm(E3_direct, axis=-1))[far]
print(f"|(-grad V) - E_direct| / |E_direct| for 4.5 cm < r < 7.5 cm: max {rel3.max():.1e}, median {np.median(rel3):.1e}")
assert rel3.max() < 0.02
print("verify passed")

## 5. Explore and rebuild

Explore. Build three systems with the same $q = 1\,$nC and spacing $d = 1\,$cm: a monopole ($+q$), a dipole ($+q$ and $-q$), and a linear quadrupole ($+q$, $-2q$, $+q$ in a line). Sample $|V|$ along a direction that is not a symmetry axis, from $r = 5d$ to $r = 500d$, and fit the slope on log-log axes. You should find $-1$, $-2$ and $-3$. Explain each in terms of cancellation: what survives at large $r$ when the lowest moment vanishes? Then repeat for $|\mathbf{E}|$ and confirm the slopes are one steeper. Finally rotate the sampling direction and watch which slopes change and which do not.

Rebuild. Close this notebook. On paper, write Coulomb's law as a vector equation and the definition of $\mathbf{E}$; derive the field on the axis of a ring and on the bisector of a finite line (worked examples 8 and 9) and take the infinite-line limit; derive the potential of a point charge from the line integral and the dipole potential at large $r$ (worked example 12). In a blank editor, write a function that returns $V$ at an $(N, 3)$ array of points from arrays of source positions and charges, and a second function that returns $\mathbf{E} = -\nabla V$ on a grid, then reproduce the checks of section 4.


## 6. Key results

| result | formula | holds when |
|---|---|---|
| Coulomb's law, force on $q_2$ from $q_1$ | $\mathbf{F} = \dfrac{q_1q_2}{4\pi\epsilon_0}\dfrac{\mathbf{r}_2 - \mathbf{r}_1}{\lvert\mathbf{r}_2 - \mathbf{r}_1\rvert^3}$ | point charges at rest |
| constants | $\epsilon_0 = 8.854\times10^{-12}\,\mathrm{C^2N^{-1}m^{-2}}$, $k = 1/4\pi\epsilon_0 = 8.988\times10^9$ | always |
| electric field | $\mathbf{E} = \mathbf{F}/q$, force on a charge $\mathbf{F} = q\mathbf{E}$ | small test charge |
| point charge | $\mathbf{E} = kq\,\mathbf{R}/R^3$, $V = kq/R$ | $V = 0$ at infinity |
| superposition | $\mathbf{E} = k\sum_i q_i\mathbf{R}_i/R_i^3$, $V = k\sum_i q_i/R_i$ | always |
| continuous body | replace $q_i$ by $\lambda\,dl'$, $\sigma\,dS'$ or $\rho\,dv'$ and sum by integrating over $\mathbf{r}'$ | always |
| ring, on axis | $E_z = kQz/(R^2 + z^2)^{3/2}$, $V = kQ/\sqrt{R^2 + z^2}$ | uniform ring |
| finite line, on bisector | $E = kQ/(d\sqrt{L^2 + d^2})$, $V = k\lambda\ln\dfrac{L + \sqrt{L^2 + d^2}}{-L + \sqrt{L^2 + d^2}}$ | uniform segment of length $2L$ |
| infinite line | $E = \lambda/2\pi\epsilon_0 d$ | $L \gg d$ |
| dipole, far field | $V = k\,\mathbf{p}\cdot\hat{\mathbf{r}}/r^2$, $E_r = 2kp\cos\theta/r^3$, $E_\theta = kp\sin\theta/r^3$ | $r \gg d$, $\mathbf{p} = q\mathbf{d}$ from $-$ to $+$ |
| potential difference | $V(\mathbf{b}) - V(\mathbf{a}) = -\int_{\mathbf{a}}^{\mathbf{b}}\mathbf{E}\cdot d\mathbf{l}$, any path | electrostatic field (curl-free) |
| potential energy | $U = qV$, work to move $q$ from $\mathbf{a}$ to $\mathbf{b}$ is $q[V(\mathbf{b}) - V(\mathbf{a})]$ | |
| field from potential | $\mathbf{E} = -\nabla V$, equipotentials perpendicular to $\mathbf{E}$ | |
| units | $\mathrm{N/C} = \mathrm{V/m}$, $1\,\mathrm{eV} = 1.602\times10^{-19}\,$J | |
| energy of a configuration | $W = k\sum_{i<j}q_iq_j/r_{ij} = \tfrac12\sum_i q_iV_i = \tfrac12\int\rho V\,dv$ | assembled from infinity |
| far-field fall-off | monopole $V \sim 1/r$, dipole $1/r^2$, quadrupole $1/r^3$; $\mathbf{E}$ one power steeper | $r \gg$ size of the system |


## 7. Warm-up answers

1. $\mathbf{F}_2 = \dfrac{q_1q_2}{4\pi\epsilon_0 R^2}\hat{\mathbf{R}}$ with $\hat{\mathbf{R}}$ pointing from $q_1$ to $q_2$. For like charges the force on $q_2$ is along $\hat{\mathbf{R}}$ (repulsion); for opposite signs the product $q_1q_2$ is negative and the force points back toward $q_1$ (attraction).
2. $\mathbf{E} = \mathbf{F}/q$, the force per unit test charge. Every electrostatic force on $q$ is proportional to $q$, so the ratio is independent of the probe, provided $q$ is small enough not to move the sources.
3. As $1/r^3$. From far away the two charges nearly coincide and their $1/r^2$ fields nearly cancel; the remainder is the difference of two fields displaced by $d$, which is smaller by a factor of order $d/r$.
4. $\mathbf{E}(\mathbf{r}) = \dfrac{1}{4\pi\epsilon_0}\displaystyle\int\rho(\mathbf{r}')\frac{\mathbf{r} - \mathbf{r}'}{|\mathbf{r} - \mathbf{r}'|^3}\,dv'$. The source point $\mathbf{r}'$ is integrated over the body; the field point $\mathbf{r}$ is fixed.
5. $V(\mathbf{b}) - V(\mathbf{a}) = -\int_{\mathbf{a}}^{\mathbf{b}}\mathbf{E}\cdot d\mathbf{l}$. The integral is the same for every path because the electrostatic field is conservative (zero curl, zero circulation), which follows from the field of a point charge being radial and superposition.
6. $V = q/4\pi\epsilon_0 r$. Far from a dipole, $V = \mathbf{p}\cdot\hat{\mathbf{r}}/4\pi\epsilon_0r^2$: it falls as $1/r^2$, is positive on the side of the positive charge, and vanishes on the plane midway between the charges.
7. $\mathbf{E} = -\nabla V$. Field lines cross equipotential surfaces at right angles, pointing from high to low $V$. The field is strongest where the equipotentials are most closely spaced.
